# NJ148 on Colab (L4): Gemma 3 1B, session B

Model `google/gemma-3-1b-it` at revision `dcc83ea8`, config `configs/training_gemma3_1b.yaml`, same data, budget and LoRA as the Llama runs.
All six runnable arms, seed 42. Results go to `MyDrive/nj148/runs_gemma3_1b/`, apart from the Llama results in `runs/`.
Needs the Gemma archive at `MyDrive/nj148/nj148_transfer_gemma.tar.gz`, and the Gemma licence accepted on Hugging Face
with the same account as the token. Outputs show counts, hashes and pass/fail only.

**Session B:** run cells 1, 2, 3 and 4 (Hugging Face login), skip 3b and 5, and start cell 7 only after session A's preflight has passed. The last cell releases the server. If the session drops, run 1, 2, 3, 4 and 7 again: finished jobs are skipped.

Session B jobs: E3·1, E3·2, E3·3, E3·4, E3·5, E9·1, E9·2, E9·3, E9·4, E9·5, E10·1, E10·2, E10·3, E10·4, E10·5.
The other session runs E1·1, E1·2, E1·3, E1·4, E1·5, E2·1, E2·2, E2·3, E2·4, E2·5, E8·1, E8·2, E8·3, E8·4, E8·5.


In [ ]:
# 1. GPU and Drive
import os, pathlib
# Colab ships TensorFlow; keep transformers from importing it (its 'no cuda drivers' message is about TensorFlow, not torch)
os.environ['USE_TF'] = '0'; os.environ['TRANSFORMERS_NO_TF'] = '1'; os.environ['PYTHONUNBUFFERED'] = '1'
import torch
assert torch.cuda.is_available(), 'no GPU: the server must be an L4'
print(torch.cuda.get_device_name(0), 'bf16', torch.cuda.is_bf16_supported())
from google.colab import drive
drive.mount('/content/drive')
DRIVE = pathlib.Path('/content/drive/MyDrive/nj148')
RUNS = DRIVE / 'runs_gemma3_1b'   # Gemma results only; the Llama results stay in runs/
for sub in ('checkpoints', 'predictions_secure', 'logs'):
    (RUNS / sub).mkdir(parents=True, exist_ok=True)
print('archive present:', (DRIVE / 'nj148_transfer_gemma.tar.gz').exists())

In [ ]:
# 2. Unpack, clone the committed code from the bundle, place gold and synthetic data
import subprocess, hashlib, json, shutil
WORK, REPO = pathlib.Path('/content/work'), pathlib.Path('/content/repo')
sh = lambda cmd: print(subprocess.run(cmd, shell=True, capture_output=True, text=True, check=True).stdout.strip())
shutil.rmtree(WORK, ignore_errors=True); shutil.rmtree(REPO, ignore_errors=True); WORK.mkdir()
sh(f'tar -xzf {DRIVE}/nj148_transfer_gemma.tar.gz -C {WORK}')
manifest = json.loads((WORK / 'transfer_manifest.json').read_text())
sh(f'git clone -q -b research-reconstruction {WORK}/repo.bundle {REPO}')
head = subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
assert head == manifest['commit'], f"commit mismatch {head} vs {manifest['commit']}"
(REPO / 'data_secure' / 'gold').mkdir(parents=True, exist_ok=True)
shutil.copy(WORK / 'gold_v2.jsonl', REPO / 'data_secure' / 'gold' / 'gold_v2.jsonl')
sha = hashlib.sha256((REPO / 'data_secure/gold/gold_v2.jsonl').read_bytes()).hexdigest()
assert sha == manifest['gold_sha256'], 'gold hash mismatch'
sh(f'cp -R {WORK}/synthetic_data/. {REPO}/synthetic_data/')
bad = [p for p, h in manifest['files'].items() if hashlib.sha256((REPO / p).read_bytes()).hexdigest() != h]
assert not bad, f'{len(bad)} files differ from the manifest'
# Official outputs must be real folders inside the repo: the code resolves paths, so a symlink to Drive would turn an
# official run into a development run. Results are copied to Drive after each step and restored from it here.
def backup():
    for sub in ('checkpoints', 'predictions_secure'):
        (REPO / sub).mkdir(exist_ok=True)
        sh(f'rsync -a {REPO}/{sub}/ {RUNS}/{sub}/')
    print('backed up to Drive')
def restore():
    for sub in ('checkpoints', 'predictions_secure'):
        (REPO / sub).mkdir(exist_ok=True)
        sh(f'rsync -a {RUNS}/{sub}/ {REPO}/{sub}/')
# macOS metadata files ('._*') would make the tree look modified and block official runs
sh(f"find {REPO} -name '._*' -print -delete | wc -l")
import sys; sys.path.insert(0, str(REPO))
from src.infrastructure.provenance import check_source_integrity
restore()
integrity = check_source_integrity(REPO)
print('tree clean for official runs:', integrity.clean, '' if integrity.clean else integrity.summary())
print('commit', head[:7], '| gold sha', sha[:8], '| files verified', len(manifest['files']))

In [ ]:
# 3. Packages pinned for Gemma 3 (needs transformers >= 4.50): the last 4.x release, tested locally with a tiny Gemma 3. transformers 5.x is NOT used.
# torch stays Colab's (no torch 2.2.2 build exists for Python 3.13 / CUDA 13); bitsandbytes 0.50.2 passed NF4 on this L4.
PINS = {"transformers": "4.57.6", "peft": "0.21.1", "accelerate": "1.14.0", "tokenizers": "0.22.2", "safetensors": "0.8.0", "huggingface_hub": "0.36.2", "bitsandbytes": "0.50.2"}
spec = ' '.join(f'{k}=={v}' for k, v in PINS.items())
!pip -q uninstall -y torchao   # peft refuses Colab's old torchao 0.10; the tested environment had no torchao
!pip -q install {spec} sentencepiece PyYAML scikit-learn pytest 2>&1 | tail -3
import importlib.metadata as md
got = {k: md.version(k) for k in PINS}
print(got, '| torch', md.version('torch'))
wrong = {k: (got[k], v) for k, v in PINS.items() if got[k] != v}
assert not wrong, f'version mismatch (installed, pinned): {wrong}'
import importlib.util
assert importlib.util.find_spec('torchao') is None, 'torchao is still installed'
!pip freeze > /content/drive/MyDrive/nj148/runs_gemma3_1b/logs/requirements-training-gemma.colab.lock
print('pins ok; lock written to Drive')

In [ ]:
# 3b. The mechanical smoke test that passed locally, on a tiny Gemma 3: tiny random model, invented data, no project data, no download
import subprocess
r = subprocess.run(['python', 'scripts/smoke_tiny_pipeline.py', '--template', 'project_gemma3_segments_v1'], cwd=REPO, capture_output=True, text=True)
print('exit', r.returncode)
print((r.stdout + r.stderr)[-3000:])

In [ ]:
# 4. Hugging Face login. The token is typed into the prompt and never stored in this notebook
from getpass import getpass
from huggingface_hub import login, whoami
login(token=getpass('Hugging Face token: '), add_to_git_credential=False)
print('logged in as', whoami()['name'])

In [ ]:
# 5. Preflight with Gemma 3 1B and the gold check. 'GPU is an RTX 3070' and the 7.5 GB limit are expected to fail on an L4
import subprocess
out = RUNS / 'logs' / 'preflight_colab_l4_gemma3_1b.json'
r = subprocess.run(['python', 'scripts/preflight_rtx3070.py', '--out', str(out), '--model-id', 'google/gemma-3-1b-it',
                    '--revision', 'dcc83ea841ab6100d6b47a070329e1ba4cf78752', '--training-config', 'configs/training_gemma3_1b.yaml', '--check-gold'],
                   cwd=REPO, capture_output=True, text=True)
print('exit', r.returncode)
print(r.stdout[-6000:])
if not out.exists():
    print(r.stderr[-2000:])


In [ ]:
# 7. OFFICIAL runs: training, then held-out inference, then backup to Drive after every step.
# Already finished jobs are skipped, so after a disconnect run cells 1-5 and this cell again. Logs: MyDrive/nj148/runs_gemma3_1b/logs/
# GEMMA 3 1B, SESSION B. Never put the same job in both sessions.
JOBS = [('E3', 1), ('E3', 2), ('E3', 3), ('E3', 4), ('E3', 5), ('E9', 1), ('E9', 2), ('E9', 3), ('E9', 4), ('E9', 5), ('E10', 1), ('E10', 2), ('E10', 3), ('E10', 4), ('E10', 5)]
CFG = 'configs/training_gemma3_1b.yaml'
SEED = 42
import glob, shutil, subprocess, time
assert check_source_integrity(REPO).clean, 'tree not clean: official runs would be refused'
def run(cmd, log):
    t0 = time.time()
    with open(log, 'a') as f:
        r = subprocess.run(['python', '-u', '-m', 'src.training.cli'] + cmd, cwd=REPO, stdout=f, stderr=subprocess.STDOUT, text=True)
    return r.returncode, round((time.time() - t0) / 60, 1)
for exp, fold in JOBS:
    run_dir = REPO / 'checkpoints' / 'official' / 'gemma3_1b' / exp / f'fold_{fold}' / f'seed_{SEED}'
    log = RUNS / 'logs' / f'{exp}_fold{fold}_seed{SEED}.log'
    if not (run_dir / 'training_manifest.json').exists():
        rc, mins = run(['train', '--experiment', exp, '--fold', str(fold), '--seed', str(SEED), '--official', '--training-config', CFG], log)
        print(exp, fold, 'train exit', rc, '| minutes', mins)
        if rc:
            print(open(log).read()[-1500:]); break
        backup()
    else:
        print(exp, fold, 'training already done')
    if glob.glob(str(REPO / 'predictions_secure' / f'{exp}__*__seed{SEED}' / f'fold_{fold}' / 'prediction_manifest.json')):
        print(exp, fold, 'inference already done'); continue
    for partial in glob.glob(str(REPO / 'predictions_secure' / f'{exp}__*__seed{SEED}' / f'fold_{fold}')):
        dest = RUNS / 'logs' / 'failed_partial' / f'{exp}_fold{fold}_{int(time.time())}'
        dest.parent.mkdir(parents=True, exist_ok=True); shutil.move(partial, dest)
        print(exp, fold, 'moved an unfinished inference folder aside')
    rc, mins = run(['infer', '--experiment', exp, '--fold', str(fold), '--seed', str(SEED), '--run', str(run_dir), '--training-config', CFG], log)
    print(exp, fold, 'infer exit', rc, '| minutes', mins)
    if rc:
        print(open(log).read()[-1500:]); break
    backup()

In [ ]:
try:
    from google.colab import runtime
    runtime.unassign()
    print('runtime released')
except Exception as e:
    print('could not release from VS Code:', type(e).__name__, '- remove the server manually: Colab: Remove Server')
